In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import gaussian_kde
from collections import defaultdict

def plot_3d_scatter_with_density(csv_folder, output_folder, intensity_col='mean_intensity_channel_2', quantile=0.75, distance_threshold=20,vol_th=100,min_intensity=1000):
    # Define common axes limits for all plots
    x_limits = [150, 350]
    y_limits = [150, 250]
    z_limits = [50, 250]

    # Collect files by odor
    files_by_odor = defaultdict(list)
    for csv_file in os.listdir(csv_folder):
        if csv_file.endswith("_cell_distances_intensities.csv"):
            odor_name = csv_file.split('60')[0]
            files_by_odor[odor_name].append(os.path.join(csv_folder, csv_file))
    # Columns representing distance values from labels
    distance_columns = ['mdG2', 'maG', 'mdG6', 'dG', 'vmG', 'lG', 'dlG', 'vpG']

    for odor_name, csv_files in files_by_odor.items():
        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        avg_centroids = {'centroid_x': [], 'centroid_y': [], 'centroid_z': [], intensity_col: []}
        for csv_path in csv_files:
            cell_data_df = pd.read_csv(csv_path)
            
            
           # Calculate the lower threshold intensity based on the quantile provided (e.g., 0.75)
            quantile_intensity_threshold = cell_data_df[intensity_col].quantile(quantile)
            # Define a minimum intensity threshold
            minimum_intensity_threshold = min_intensity
            # Use the maximum between the quantile threshold and the minimum threshold
            lower_intensity_threshold = max(quantile_intensity_threshold, minimum_intensity_threshold)
            # Calculate the upper threshold intensity based on the 0.99 quantile
            upper_intensity_threshold = cell_data_df[intensity_col].quantile(0.99)
            # Filtering based on intensity within the thresholds and distance values
            filtered_data_df = cell_data_df[
                (cell_data_df[intensity_col] >= lower_intensity_threshold) &
                (cell_data_df[intensity_col] < upper_intensity_threshold) &  # Exclude values in the top 0.99 quantile
                (cell_data_df[distance_columns].min(axis=1) <= distance_threshold) &
                (cell_data_df['volume'] >= vol_th)

            ]

            
            
            
            # # Determine the threshold intensity based on the specified quantile
            # intensity_threshold = cell_data_df[intensity_col].quantile(quantile)
            # # Filter based on intensity value
            # filtered_data_df = cell_data_df[(cell_data_df[intensity_col] >= intensity_threshold) & 
            #                 (cell_data_df[distance_columns].min(axis=1) <= distance_threshold)]
            avg_centroids['centroid_x'].extend(filtered_data_df['centroid_x'])
            avg_centroids['centroid_y'].extend(filtered_data_df['centroid_y'])
            avg_centroids['centroid_z'].extend(filtered_data_df['centroid_z'])
            avg_centroids[intensity_col].extend(filtered_data_df[intensity_col])

        # Scatter plot with color according to intensity
        scatter = ax.scatter(avg_centroids['centroid_x'], avg_centroids['centroid_y'], avg_centroids['centroid_z'], alpha=0.8, 
                             c=avg_centroids[intensity_col], cmap='plasma', s=10,vmax=2500)#,vmin=1000, vmax=2000)

        # Creating the density plots
        x, y, z = np.mgrid[x_limits[0]:x_limits[1]:30j, y_limits[0]:y_limits[1]:30j, z_limits[0]:z_limits[1]:30j]
        positions = np.vstack([x.ravel(), y.ravel(), z.ravel()])
        density = gaussian_kde(np.vstack([avg_centroids['centroid_x'], avg_centroids['centroid_y'], avg_centroids['centroid_z']]))
        values = density(positions)
        density_values = values.reshape(x.shape)

        # Z-axis projection
        ax.contour(x[:, :, 0], y[:, :, 0], np.sum(density_values, axis=2), zdir='z', offset=z_limits[0], cmap='plasma', alpha=0.9)
        # Y-axis projection
        ax.contour(x[:, :, 0], np.sum(density_values, axis=1), z[0, :, :], zdir='y', offset=y_limits[1], cmap='plasma', alpha=0.9)
        # X-axis projection
        ax.contour(np.sum(density_values, axis=0), y[0, :, :], z[0, :, :], zdir='x', offset=x_limits[0], cmap='plasma', alpha=0.9)

        # Common settings, colorbar, and display
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)
        ax.set_zlim(z_limits)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
        
        num_files = len(csv_files)
        #plt.title(f"3D Scatter and Density Plot for {odor_name} (n={num_files})\nwith distance of less than {distance_threshold}um")
        plt.title(f" {odor_name}\n (n={num_files})", fontsize=30)#\nwith distance of less than {distance_threshold}um")
        # Modified colorbar
        colorbar = fig.colorbar(scatter, ax=ax, orientation="vertical", label='Activity A.U.', shrink=0.5)
        colorbar.ax.set_position([.85, .25, .05, .5]) # Adjust the position and size of colorbar

        output_path = os.path.join(output_folder, f"3D Scatter and Density Plot for {odor_name} with distance of less than {distance_threshold}um.png")
        output_path_svg = os.path.join(output_folder, f"3D Scatter and Density Plot for {odor_name} with distance of less than {distance_threshold}um.svg")

        plt.tight_layout()
        plt.savefig(output_path)
        plt.savefig(output_path_svg,format='svg')
        
        plt.show()


# Example call to the function
#csv_folder = r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\spinSR\20250120_CaMPARI\tiff_files\reg_files\merged_files\output\expanded_masks\output'
#output_folder = csv_folder + os.sep + 'output_plots'
if not os.path.exists(output_folder):
    os.mkdir(output_folder)
plot_3d_scatter_with_density(csv_folder, output_folder, quantile=0.1, distance_threshold=30,vol_th=100,min_intensity=0)

